# 02 — Dataclasses

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/02_dataclasses.ipynb)

## 📌 What is it?
`@dataclass` auto-generates `__init__`, `__repr__`, `__eq__`, and other methods from class annotations.

## 🧠 Why AI Engineers need this
Model configs, API request/response schemas, dataset samples, and training hyperparameters are all naturally represented as dataclasses.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import Optional

@dataclass
class TrainingConfig:
    # Required fields (no default)
    model_name: str
    dataset_path: str
    
    # Optional fields with defaults
    learning_rate: float = 3e-4
    batch_size: int = 32
    epochs: int = 10
    max_seq_len: int = 512
    warmup_steps: int = 100
    gradient_clip: float = 1.0
    use_fp16: bool = True
    output_dir: str = "./checkpoints"
    
    # Mutable defaults MUST use field()
    tags: list = field(default_factory=list)
    
    def __post_init__(self):
        """Validation after __init__."""
        if self.learning_rate <= 0:
            raise ValueError(f"learning_rate must be > 0, got {self.learning_rate}")
        if self.batch_size <= 0:
            raise ValueError(f"batch_size must be > 0")

cfg = TrainingConfig(
    model_name="bert-base-uncased",
    dataset_path="/data/train.jsonl",
    learning_rate=5e-5,
    epochs=3
)
print(cfg)   # auto __repr__
print(cfg.learning_rate)
print(asdict(cfg))   # convert to dict for JSON

In [ ]:
# ── FROZEN DATACLASSES (immutable) ──
from dataclasses import dataclass

@dataclass(frozen=True)   # immutable after creation
class ModelID:
    provider: str
    name: str
    version: str
    
    def __str__(self):
        return f"{self.provider}/{self.name}@{self.version}"

gpt4 = ModelID("openai", "gpt-4o", "2024-11-20")
print(gpt4)         # openai/gpt-4o@2024-11-20
print(hash(gpt4))   # hashable because frozen!

# Frozen dataclasses can be used as dict keys!
model_costs = {
    gpt4: 0.0025,
    ModelID("anthropic", "claude-3-5-sonnet", "20241022"): 0.003
}
print(model_costs[gpt4])

# Trying to modify raises error
try:
    gpt4.name = "gpt-5"
except Exception as e:
    print(f"Immutable! {e}")

In [ ]:
# ── INHERITANCE with dataclasses ──
from dataclasses import dataclass

@dataclass
class BaseConfig:
    verbose: bool = False
    seed: int = 42

@dataclass
class FineTuneConfig(BaseConfig):
    base_model: str = "bert-base-uncased"
    num_labels: int = 2
    freeze_base: bool = True
    
cfg = FineTuneConfig(verbose=True, base_model="roberta-base", num_labels=5)
print(cfg)

## ⚠️ Common Mistakes
```python
# ❌ Mutable default without field()
@dataclass
class Bad:
    items: list = []     # ValueError at class creation!

# ✅
@dataclass
class Good:
    items: list = field(default_factory=list)

# ❌ Inheriting from non-dataclass with defaults
# Fields with defaults must come AFTER fields without defaults
@dataclass
class Bad:
    x: int = 0    # default
    y: int        # NO default — SyntaxError!
```

## 🔗 What's Next?
[03 — Async/Await →](03_async_and_await.ipynb)